# P6 - Two-model ensemble (HistGradientBoostingRegressor + LightGBM)

Source: DrivenData "Water Supply Forecast Rodeo" winner solutions repo (see RESOURCES.md) - both 1st and 2nd place production pipelines for a comparable hydrological-forecasting problem combine multiple model fits, not a single one. A first attempt at the cheapest possible ensembling form - averaging several seeds of the *same* `HistGradientBoostingRegressor` model - showed no real signal (2/5 seeds, -0.03% mean, noise-level; see git history for that ruled-out attempt). This notebook tries genuine model diversity instead: combine `HistGradientBoostingRegressor` with LightGBM, a different gradient-boosting implementation with its own tree-growth/split-finding algorithm (also the model family used by the M5 Forecasting 1st place solution already cited for P5), and average their predictions.

Both models get identical feature inputs (the same raw NaN-containing feature matrix - LightGBM has native missing-value handling too, so this doesn't undo P5's finding that missingness itself is informative).

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np

from src import config, data, evaluate, features, model
from src.train import get_feature_cols

raw_train, test, sample_submission = data.load_raw_data()
target_horizons = evaluate.compute_test_horizons(
    features.build_all_features(raw_train, test)[0], test
)
masked_month_fraction, masked_row_fraction = evaluate.measure_masking_pattern(test)
print("target_horizons:", sorted(target_horizons))
print("masked_month_fraction:", masked_month_fraction, "masked_row_fraction:", masked_row_fraction)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



target_horizons: [1, 5, 6, 7, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 35, 39, 40]
masked_month_fraction: 0.6666666666666666 masked_row_fraction: 0.9977735834478643


In [2]:
histgbr_rmses = []
lightgbm_rmses = []
ensemble_rmses = []

for masking_seed in range(5):
    fit_df, val_df = evaluate.mask_augmented_horizon_matched_split(
        raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
        fit_seed=masking_seed, val_seed=masking_seed + 100,
    )
    feature_cols = get_feature_cols(fit_df)
    X_fit = features.select_base_features(fit_df, feature_cols)
    y_fit = fit_df[config.TARGET_COL].to_numpy()
    X_val = features.select_base_features(val_df, feature_cols)
    y_val = val_df[config.TARGET_COL].to_numpy()

    histgbr = model.make_baseline_model()
    histgbr.fit(X_fit, y_fit)
    histgbr_pred = model.predict(histgbr, X_val)
    histgbr_rmse = evaluate.rmse(y_val, histgbr_pred)
    histgbr_rmses.append(histgbr_rmse)

    lgbm = model.make_lightgbm_model()
    lgbm.fit(X_fit, y_fit)
    lgbm_pred = model.predict(lgbm, X_val)
    lgbm_rmse = evaluate.rmse(y_val, lgbm_pred)
    lightgbm_rmses.append(lgbm_rmse)

    ensemble_pred = model.predict_ensemble([histgbr, lgbm], X_val)
    ensemble_rmse = evaluate.rmse(y_val, ensemble_pred)
    ensemble_rmses.append(ensemble_rmse)

    print(f"masking_seed={masking_seed}: histgbr={histgbr_rmse:.4f}  lightgbm={lgbm_rmse:.4f}  "
          f"ensemble={ensemble_rmse:.4f}  delta_vs_histgbr={ensemble_rmse - histgbr_rmse:+.4f}")


C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _wina

masking_seed=0: histgbr=0.7235  lightgbm=0.7249  ensemble=0.7238  delta_vs_histgbr=+0.0003


masking_seed=1: histgbr=0.7212  lightgbm=0.7197  ensemble=0.7200  delta_vs_histgbr=-0.0012


masking_seed=2: histgbr=0.7113  lightgbm=0.7123  ensemble=0.7113  delta_vs_histgbr=+0.0001


masking_seed=3: histgbr=0.6926  lightgbm=0.6950  ensemble=0.6934  delta_vs_histgbr=+0.0008


masking_seed=4: histgbr=0.7021  lightgbm=0.7020  ensemble=0.7017  delta_vs_histgbr=-0.0004


In [3]:
def summarize(name, values):
    m = sum(values) / len(values)
    print(f"{name:10s} mean RMSE: {m:.4f}  (range {min(values):.4f}-{max(values):.4f})")
    return m

histgbr_mean = summarize("HistGBR", histgbr_rmses)
lgbm_mean = summarize("LightGBM", lightgbm_rmses)
ensemble_mean = summarize("Ensemble", ensemble_rmses)

wins = sum(e < s for e, s in zip(ensemble_rmses, histgbr_rmses))
print(f"\nEnsemble wins {wins}/5 seeds vs. HistGBR alone, "
      f"mean delta {ensemble_mean - histgbr_mean:+.4f} ({(ensemble_mean - histgbr_mean) / histgbr_mean:+.2%})")


HistGBR    mean RMSE: 0.7101  (range 0.6926-0.7235)
LightGBM   mean RMSE: 0.7108  (range 0.6950-0.7249)
Ensemble   mean RMSE: 0.7100  (range 0.6934-0.7238)

Ensemble wins 2/5 seeds vs. HistGBR alone, mean delta -0.0001 (-0.01%)


## Gate decision

**Not graduated - confirmed negative result, no real submission needed.**

Ensemble mean RMSE 0.7100 vs. HistGBR-alone mean 0.7101 (-0.01%), winning only 2/5 masking realisations (not a majority) - even weaker than the same-model seed-averaging attempt (-0.03%, also 2/5). LightGBM standalone (mean 0.7108) is itself very slightly worse than HistGBR, not complementary enough to move the average: the two models' errors are evidently too correlated (identical feature set, identical boosting-lever budget, identical training data) for averaging to help here. This is an order of magnitude weaker than P2/P5's real weak-but-consistently-signed cases (~0.1-0.23% mean, 3/5 seeds) - noise, not a signal.

Likely explanation: the DrivenData winning solutions combine models trained on genuinely different feature sets/targets (a seasonal model + a monthly model), not the same features fed to two boosting libraries with matched hyperparameters. Real diversity there comes from problem decomposition, not just algorithm choice - which is a much bigger undertaking than this experiment, and not attempted here given the ~5 days left before close.

`src/model.py::make_lightgbm_model`/`predict_ensemble` stay committed but unused (not wired into `src/train.py`), available for later reuse.